# Update 1 — Same-condition comparison with the Mathar model

**Revision note (v2.0):** This notebook re-derives the comparison between the measured environmental coefficients and the Mathar model **on identical data rows**, replacing the earlier fixed-condition comparison that produced the retired "+17.2 % humidity enhancement" claim.

A linear surrogate model

$$n = n_0 + \alpha_T\,T + \alpha_H\,H + \alpha_P\,P$$

is fitted to (i) the measured refractive index and (ii) the Mathar formulation evaluated at the identical observed $(T,H,P)$ rows. Both fits share the same design matrix; the coefficient difference is estimated with a **paired moving-block bootstrap** that accounts for the correlation between the two estimates.

**Key numbers reproduced (cf. Supplemental Material, Table S2):**

| Analysis | N | $\Delta\alpha_T$ | $\Delta\alpha_H$ | $\Delta\alpha_P$ |
|---|---|---|---|---|
| In-domain (10–25 °C) | 12,134 | $-1.41\times10^{-8}$ | $+4.62\times10^{-9}$ | $+3.79\times10^{-9}$ |
| Full campaign | 145,784 | $-3.11\times10^{-9}$ | $+4.33\times10^{-9}$ | $+2.39\times10^{-9}$ |

Fraction of observations above 25 °C: 91.7 %. The difference is resampled over **physical-time blocks** with the duration swept from 0.56 h to 48 h and the coverage fraction reported for each pool; the descriptive residual-ACF 1/e lag is $\tau$ = 78 samples, but it is **not** adopted as a block-length rule because a diurnal lobe persists in the residual ACF. B = 5000 replications.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf
import sys, os, time
sys.path.append('..')

from models.mathar.Mathar2007 import n as n_mathar_scalar

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})

In [2]:
# Load data (columns: time, counts_ratio, humidity, temperature, pressure, n_1762)

df = pd.read_csv('../../data/processed/full_data.csv')
df = df.sort_values('time').reset_index(drop=True)   # ensure temporal order
print(f"N = {len(df)}")
print(df[['time','temperature','humidity','pressure','n_1762']].describe())

T_C = df['temperature'].values     # deg C
H_pct = df['humidity'].values      # %RH
P_hPa = df['pressure'].values      # hPa
n_data = df['n_1762'].values       # absolute refractive index


N = 145784
         temperature       humidity       pressure         n_1762
count  145784.000000  145784.000000  145784.000000  145784.000000
mean       28.874692      30.230868     986.403238       1.000254
std         3.380446       3.856421       4.481461       0.000003
min        17.518200      18.935562     964.605044       1.000248
25%        28.055459      27.315783     984.418391       1.000253
50%        29.446097      29.707467     987.300861       1.000254
75%        30.914016      33.085634     989.062836       1.000255
max        34.622549      45.320557    1006.774328       1.000270


In [3]:
# Mathar cloud evaluated at the identical (T, H, P) rows

lam_um = 1.762 # unit: µm
T_K = T_C + 273.15 # unit: K
P_Pa = P_hPa * 100.0 # unit: Pa

t0 = time.time()
n_mathar = np.array([n_mathar_scalar(lam_um, Tk, pp, hh)
                     for Tk, pp, hh in zip(T_K, P_Pa, H_pct)])
print(f"Mathar cloud generated in {time.time()-t0:.1f} s")
print(f"n_mathar range: [{n_mathar.min():.12f}, {n_mathar.max():.12f}]")


Mathar cloud generated in 1.4 s
n_mathar range: [1.000247321467, 1.000268634383]


In [4]:
# Analysis windows
mask_in  = (T_C >= 10.0) & (T_C <= 25.0)          # Mathar validity domain
mask_full = np.ones(len(df), dtype=bool)

n_above25 = int((~mask_in).sum())
print(f"In-domain (10-25 C): N = {mask_in.sum()}")
print(f"Above 25 C:          N = {n_above25}  ({100*n_above25/len(df):.1f} %)")


# Linear surrogate fit (OLS via normal equations; cross-checked vs statsmodels)
def fit_surrogate(idx):
    X = np.column_stack([np.ones(len(idx)), T_C[idx], H_pct[idx], P_hPa[idx]])
    beta, *_ = np.linalg.lstsq(X, n_data[idx], rcond=None)
    return beta                     # [n0, aT, aH, aP]

def fit_surrogate_mathar(idx):
    X = np.column_stack([np.ones(len(idx)), T_C[idx], H_pct[idx], P_hPa[idx]])
    beta, *_ = np.linalg.lstsq(X, n_mathar[idx], rcond=None)
    return beta

# Cross-check with statsmodels on the full sample
idx_full = np.arange(len(df))
Xf = np.column_stack([np.ones(len(df)), T_C, H_pct, P_hPa])
sm_res = sm.OLS(n_data, Xf).fit()
beta_manual = fit_surrogate(idx_full)
print("Sanity check (manual vs statsmodels, full sample):")
print("  manual  :", beta_manual)
print("  statsmodels:", sm_res.params)
print("  max abs diff:", np.abs(beta_manual - sm_res.params).max())


In-domain (10-25 C): N = 12134
Above 25 C:          N = 133650  (91.7 %)
Sanity check (manual vs statsmodels, full sample):
  manual  : [ 1.00002431e+00 -8.84742454e-07 -1.31524232e-08  2.59490560e-07]
  statsmodels: [ 1.00002431e+00 -8.84742454e-07 -1.31524232e-08  2.59490560e-07]
  max abs diff: 2.531308496145357e-14


In [5]:
# Point estimates: data vs Mathar surrogate

idx_in = np.where(mask_in)[0]

def report(name, bd, bm):
    print(f"--- {name} ---")
    print(f"  data   :  aT={bd[1]:.10e}  aH={bd[2]:.10e}  aP={bd[3]:.10e}")
    print(f"  mathar :  aT={bm[1]:.10e}  aH={bm[2]:.10e}  aP={bm[3]:.10e}")
    print(f"  Delta  : daT={bd[1]-bm[1]:+.6e}  daH={bd[2]-bm[2]:+.6e}  daP={bd[3]-bm[3]:+.6e}")
    print(f"  Delta %: daT={100*(bd[1]-bm[1])/abs(bm[1]):+.2f}%  daH={100*(bd[2]-bm[2])/abs(bm[2]):+.2f}%  daP={100*(bd[3]-bm[3])/abs(bm[3]):+.2f}%")

report("Full campaign", fit_surrogate(idx_full), fit_surrogate_mathar(idx_full))
report("In-domain (10-25 C)", fit_surrogate(idx_in), fit_surrogate_mathar(idx_in))


--- Full campaign ---
  data   :  aT=-8.8474245433e-07  aH=-1.3152423152e-08  aP=2.5949055957e-07
  mathar :  aT=-8.8163622117e-07  aH=-1.7485824221e-08  aP=2.5709600246e-07
  Delta  : daT=-3.106233e-09  daH=+4.333401e-09  daP=+2.394557e-09
  Delta %: daT=-0.35%  daH=+24.78%  daP=+0.93%
--- In-domain (10-25 C) ---
  data   :  aT=-9.3015150391e-07  aH=-6.7936612090e-09  aP=2.6935203329e-07
  mathar :  aT=-9.1606809452e-07  aH=-1.1415687397e-08  aP=2.6555829221e-07
  Delta  : daT=-1.408341e-08  daH=+4.622026e-09  daP=+3.793741e-09
  Delta %: daT=-1.54%  daH=+40.49%  daP=+1.43%


In [6]:
# Residual autocorrelation: descriptive only. A 1/e-based block rule is NOT used.
#
# The residual ACF decays fast but keeps a positive diurnal lobe (~0.14 at
# 20-22 h; see 06_statistical_robustness.ipynb), so no single decorrelation
# scale describes the full dependence and L = 2*tau_1e is not adopted.

from physical_blocks import (block_grid_summary, build_blocks, coverage,
                             segment_bounds, time_seconds)

beta_full = fit_surrogate(idx_full)
resid = n_data - Xf @ beta_full
acf_vals = acf(resid, nlags=min(4000, len(resid) // 4))
# NB: this ACF is evaluated along the row axis of the whole file, so it
# includes pairs that span the 111-day gap.  Evaluated within contiguous
# segments instead (06_statistical_robustness.ipynb) the 1/e crossing is
# ~100 samples (~35 min).  Neither value is adopted as a block rule.
tau_1e = next(l for l in range(1, len(acf_vals)) if acf_vals[l] < 1 / np.e)
print('Descriptive whole-file 1/e lag of the residual ACF:', tau_1e,
      'samples (NOT adopted as a block rule; see note above)')

t_sec = time_seconds(df['time'])
seg = segment_bounds(t_sec)
print('continuous segments:', len(seg), '| largest gap (days):',
      round(np.diff(t_sec).max() / 86400, 1))

print()
print('Physical-time block grid (no block crosses a data gap):')
print('     D (h)  n_blocks    n_used  coverage')
for row in block_grid_summary(t_sec):
    print('%9.2f %9d %9d %9.3f' % (row['D_hours'], row['n_blocks'],
                                   row['n_used'], row['coverage']))


Descriptive whole-file 1/e lag of the residual ACF: 78 samples (NOT adopted as a block rule; see note above)
continuous segments: 104 | largest gap (days): 111.3

Physical-time block grid (no block crosses a data gap):
     D (h)  n_blocks    n_used  coverage
     0.56      1379    145196     0.996
     7.17       122    136468     0.936
    24.00        31     84570     0.580
    48.00         5     21881     0.150


In [7]:
# Paired moving-block bootstrap over PHYSICAL-TIME blocks
# (supersedes the fixed L = 156 row blocks, which could straddle the 111-day gap)
# Pre-aggregated per-block moments -> vectorized resampling

from physical_blocks import (D_LIST_HOURS, coverage as block_coverage,
                            paired_coefficient_bootstrap)

# Single source of truth for the swept durations: 0.559 / 7.167 / 24 / 48 h,
# derived inside physical_blocks from D_LIST_SECONDS (2012.4 / 25800 / 86400 /
# 172800 s).  Do not re-declare the durations here.
D_SWEEP = D_LIST_HOURS

def paired_phys_bootstrap(mask, D_hours, B=5000, seed=42):
    '''Paired bootstrap of beta_data - beta_mathar on physical-time blocks.
    Returns (diffs, coverage, n_blocks); no block crosses a data gap.'''
    idx = np.where(mask)[0]
    t_sub = t_sec[idx]
    seg_sub = segment_bounds(t_sub)
    blocks = build_blocks(t_sub, D_hours * 3600.0, seg_sub)
    diffs = paired_coefficient_bootstrap(Xf[idx], n_data[idx], n_mathar[idx],
                                         blocks, n_draws=B, seed=seed)
    return diffs, block_coverage(blocks, len(idx)), len(blocks)

def boot_report(name, diffs, cov, nbl):
    print(f'--- {name} | {nbl} blocks | coverage {cov:.3f} ---')
    for j, lab in enumerate(['aT', 'aH', 'aP']):
        col = diffs[:, j + 1]
        lo, hi = np.percentile(col, [2.5, 97.5])
        print(f'  Delta_{lab}: SE {col.std(ddof=1):.3e}   95% CI [{lo:+.4e}, {hi:+.4e}]')

t0 = time.time()
boot = {}
for D in D_SWEEP:
    boot[('full', D)] = paired_phys_bootstrap(np.ones(len(df), dtype=bool), D)
    boot[('in', D)] = paired_phys_bootstrap(mask_in, D)
print(f'Bootstrap sweep completed in {time.time()-t0:.0f} s')

for D in D_SWEEP:
    boot_report(f'In-domain, D = {D} h', *boot[('in', D)])
for D in D_SWEEP:
    boot_report(f'Full campaign, D = {D} h', *boot[('full', D)])


Bootstrap sweep completed in 1 s
--- In-domain, D = 0.559 h | 271 blocks | coverage 0.998 ---
  Delta_aT: SE 1.302e-08   95% CI [-3.9864e-08, +1.1378e-08]
  Delta_aH: SE 4.631e-09   95% CI [-3.7995e-09, +1.4434e-08]
  Delta_aP: SE 1.326e-09   95% CI [+1.3093e-09, +6.5291e-09]
--- In-domain, D = 7.166666666666667 h | 26 blocks | coverage 0.992 ---
  Delta_aT: SE 3.415e-08   95% CI [-8.6632e-08, +5.2381e-08]
  Delta_aH: SE 1.115e-08   95% CI [-1.4651e-08, +2.8880e-08]
  Delta_aP: SE 3.631e-09   95% CI [-3.2896e-09, +1.1088e-08]
--- In-domain, D = 24.0 h | 8 blocks | coverage 0.948 ---
  Delta_aT: SE 5.395e-08   95% CI [-1.2091e-07, +9.4497e-08]
  Delta_aH: SE 1.357e-08   95% CI [-2.0021e-08, +3.5262e-08]
  Delta_aP: SE 5.073e-09   95% CI [-5.5886e-09, +1.4080e-08]
--- In-domain, D = 48.0 h | 3 blocks | coverage 0.919 ---
  Delta_aT: SE 3.042e-08   95% CI [-9.5725e-08, +6.8511e-08]
  Delta_aH: SE 1.924e-08   95% CI [-2.2546e-08, +4.7085e-08]
  Delta_aP: SE 5.307e-09   95% CI [-8.2303e-09,

In [8]:
# Within-campaign blocked cross-validation (K=5 folds over physical-time blocks)
#
# The blocks are assigned to folds across the whole campaign, so the held-out
# blocks are interleaved with the training blocks rather than separated in time.
# This is an internal consistency check, NOT a temporally held-out validation
# (cf. Roberts et al. 2017 for a temporally separated scheme).

from physical_blocks import build_blocks

def within_campaign_cv(y, D_hours, K=5, seed=42):
    blocks = build_blocks(t_sec, D_hours * 3600.0, seg)
    rng = np.random.default_rng(seed)
    fold = rng.permutation(np.arange(len(blocks)) % K)
    rmse = []
    for k in range(K):
        te = np.concatenate([np.arange(blocks[b][0], blocks[b][1] + 1)
                             for b in range(len(blocks)) if fold[b] == k])
        tr = np.concatenate([np.arange(blocks[b][0], blocks[b][1] + 1)
                             for b in range(len(blocks)) if fold[b] != k])
        beta, *_ = np.linalg.lstsq(Xf[tr], y[tr], rcond=None)
        rmse.append(np.sqrt(np.mean((y[te] - Xf[te] @ beta) ** 2)))
    return np.array(rmse)

D_CV_H = 1.0
rmse_data   = within_campaign_cv(n_data, D_CV_H)
rmse_mathar = within_campaign_cv(n_mathar, D_CV_H)
print(f'Within-campaign blocked CV (D = {D_CV_H} h, K = 5)')
print(f'Data surrogate CV RMSE   : {rmse_data.mean():.4e}  +/- {rmse_data.std():.2e}')
print(f'Mathar surrogate CV RMSE : {rmse_mathar.mean():.4e}  +/- {rmse_mathar.std():.2e}')


Within-campaign blocked CV (D = 1.0 h, K = 5)
Data surrogate CV RMSE   : 1.8423e-07  +/- 1.58e-09
Mathar surrogate CV RMSE : 4.4768e-08  +/- 1.51e-09
